# Kodak-24: common benchmark for every reconstruction model

This is the team's single local evaluation entry point. It uses all 24 native-resolution Kodak images and produces one CSV row per method × operating point × image. Built-in RAW RGB24 and JPEG baselines use exactly the same timing and quality code as learned models.

**Metric rules:** PSNR/SSIM are RGB `[0,1]`; LPIPS is AlexNet; CBR is complex channel uses divided by `3×H×W`; compression ratio is raw RGB24 bits divided by serialized payload bits. JPEG/RAW do not claim a CBR unless a digital PHY is explicitly supplied. No evaluation-side resizing is allowed.

## 1. Environment

Create the repository's Python 3.12 environment and install `requirements.txt` before opening this notebook. If this kernel is new, run `%pip install -e ..[benchmark,metrics]` once and restart it. LPIPS may download its pretrained AlexNet weights on first use.

In [ ]:
from pathlib import Path
import getpass, math, random, subprocess, sys

import numpy as np
import pandas as pd
from PIL import Image
import torch

candidate = Path.cwd().resolve()
PROJECT_ROOT = candidate if (candidate / 'pyproject.toml').is_file() else candidate.parent
if not (PROJECT_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Start Jupyter in the repository root or its Notebooks directory.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from Evaluation.kodak_benchmark import (
    CallableCodec, EncDecCodec, JPEGCodec, RawRGBCodec, append_rows_csv,
    benchmark_codec, dataset_files, environment_summary, make_run_id,
    summarize_rows,
)
environment_summary()

In [ ]:
# Team/run configuration. Keep one RUN_ID for one complete execution.
CONTRIBUTOR = getpass.getuser()             # Prefer your real team identifier.
RUN_ID = make_run_id()                      # Unique, merge-safe measurement identity.
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
KODAK_DIR = PROJECT_ROOT / 'data' / 'Kodak24'
CSV_PATH = PROJECT_ROOT / 'results' / 'kodak24_benchmark.csv'
JPEG_QUALITIES = [20, 30, 40, 50, 60, 70, 80, 90]
TIMING_REPETITIONS = 3
WARMUP_REPETITIONS = 1
COMPUTE_LPIPS = True
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print({'contributor': CONTRIBUTOR, 'run_id': RUN_ID, 'device': DEVICE})

## 2. Download and audit Kodak-24

The downloader resumes valid files, downloads through `.part` files, checks the published byte count and canonical SHA-256, validates each RGB PNG header and dimension, and writes a local manifest. This notebook then decodes every image with Pillow. The dataset directory is intentionally ignored by Git.

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / 'scripts' / 'download_kodak24.py'), '--output', str(KODAK_DIR)],
    check=True, cwd=PROJECT_ROOT,
)
KODAK_FILES = dataset_files(KODAK_DIR)
audit = []
for path in KODAK_FILES:
    with Image.open(path) as image:
        audit.append({'image_id': path.stem, 'width': image.width, 'height': image.height, 'mode': image.mode})
audit_df = pd.DataFrame(audit)
assert len(audit_df) == 24 and set(audit_df['mode']) == {'RGB'}
display(audit_df)
display(audit_df.groupby(['width', 'height']).size().rename('images').reset_index())

## 3. Register methods

RAW and JPEG are ready. Add each learned model either through `EncDecCodec` (preferred for repository adapters) or `CallableCodec`. Give every checkpoint/rate/SNR a distinct `operating_point` and a checkpoint hash or version string.

For continuous JSCC, report **complex channel uses**, not bytes. For an actually serialized latent, provide `payload_bits_fn`. If a number is not meaningful, leave it empty instead of inventing a conversion.

In [ ]:
METHODS = [RawRGBCodec()] + [JPEGCodec(quality) for quality in JPEG_QUALITIES]

# Preferred: a repository EncDecInterface adapter that is already built/loaded.
# channel = AWGNChannel()
# snr_db, rate = 10.0, 96
# METHODS.append(EncDecCodec(
#     name='MySwinJSCC', adapter=my_adapter, device=DEVICE,
#     version='checkpoint-sha256-prefix',
#     operating_point=f'rate={rate};snr_db={snr_db};snr=Es/N0',
#     encode_options={'rate': rate, 'snr_db': snr_db},
#     transport_fn=lambda latent: channel.transmit(latent, snr_db),
# ))

# Generic PyTorch example. encode_fn/decode_fn are the only required hooks.
# my_model = load_my_model(...).to(DEVICE).eval()
# METHODS.append(CallableCodec(
#     name='MyModel', version='checkpoint-sha256-prefix',
#     operating_point='rate=example;snr_db=10', device=DEVICE,
#     encode_fn=lambda x: my_model.encoder(x),
#     decode_fn=lambda z: my_model.decoder(z),
#     channel_uses_fn=lambda z: math.ceil(z[0].numel() / 2),
#     modules=[my_model],
#     flops=None,  # Fill from your model's trusted profiler if available.
# ))

keys = [(method.name, method.operating_point) for method in METHODS]
assert len(keys) == len(set(keys)), 'Duplicate method/operating-point key'
pd.DataFrame(keys, columns=['method', 'operating_point'])

### DeepJSCC example with the repository adapter

Uncomment and edit this example on the teammate machine that has the external source/checkpoint. Models that only accept tiles should wrap tiling inside their callable hooks; the benchmark will reject a final image whose size differs from the original.

In [ ]:
# from EncDecPipeline.Models.DeepJSCC.adapter import DeepJSCCAdapter
# from EncDecPipeline.Models.DeepJSCC.channel_wrapper import DeepJSCCChannelWrapper
# DEEP_ROOT = Path(r'D:/path/to/Deep-JSCC-PyTorch')
# CHECKPOINT = DEEP_ROOT / 'out' / 'checkpoint.pth'
# deep = DeepJSCCAdapter(CHECKPOINT, DEEP_ROOT, c=19, device=DEVICE).build()
# awgn = DeepJSCCChannelWrapper()
# snr_db = 10.0
# METHODS.append(EncDecCodec(
#     'DeepJSCC', deep, device=DEVICE,
#     version=deep.parameter_summary()['checkpoint_sha256'],
#     operating_point=f'c=19;snr_db={snr_db};snr=Es/N0',
#     transport_fn=lambda latent: awgn.transmit(latent, snr_db),
# ))

## 4. Run every method on all 24 images

This can take time, especially LPIPS on CPU. GPU timing is synchronized. Memory columns report process RSS and CUDA peak allocation; parameter/storage columns report static model complexity. FLOPs remain empty unless the model owner supplies a trusted value.

In [ ]:
all_rows = []
for method in METHODS:
    method_rows = benchmark_codec(
        method, KODAK_FILES, run_id=RUN_ID, contributor=CONTRIBUTOR,
        project_root=PROJECT_ROOT, timing_repetitions=TIMING_REPETITIONS,
        warmup_repetitions=WARMUP_REPETITIONS, compute_lpips=COMPUTE_LPIPS,
    )
    assert len(method_rows) == 24
    all_rows.extend(method_rows)

per_image_df = pd.DataFrame(all_rows)
counts = per_image_df.groupby(['method', 'operating_point'])['image_id'].nunique()
assert (counts == 24).all(), f'Incomplete method results:\n{counts[counts != 24]}'
display(per_image_df)

In [ ]:
summary_df = pd.DataFrame(summarize_rows(all_rows))
columns = [
    'method', 'operating_point', 'images', 'compression_ratio_mean',
    'bits_per_pixel_mean', 'cbr_mean', 'encode_time_ms_mean',
    'decode_time_ms_mean', 'cpu_process_time_ms_mean',
    'peak_cpu_rss_delta_mb_mean', 'peak_gpu_allocated_mb_mean',
    'model_parameters_mean', 'model_size_mb_mean', 'flops_mean',
    'psnr_db_mean', 'ssim_mean', 'lpips_mean',
]
display(summary_df[columns].sort_values(['method', 'operating_point']))

## 5. Append to the shared CSV

Review the tables first. Then set the switch to `True` and run once. The writer is schema-checked, locked, atomic, and idempotent for the same run/method/operating-point/image key. Pull/rebase main before running and again before pushing, because Git itself can still report a CSV merge conflict.

In [ ]:
APPEND_RESULTS = False
if APPEND_RESULTS:
    added = append_rows_csv(CSV_PATH, all_rows)
    print(f'Appended {added} new rows to {CSV_PATH}')
else:
    print('Review complete? Set APPEND_RESULTS=True to persist these rows.')

## Interpretation checklist

- Compare quality at matched rate/resource points, not only the highest PSNR.
- RAW must be lossless (`PSNR=∞`, `SSIM=1`, `LPIPS≈0`, compression ratio `1`).
- JPEG has payload bits/BPP/compression ratio but no CBR without an explicit modulation/coding chain.
- Analog JSCC normally has channel uses/CBR but no payload bits.
- Do not compare CPU and GPU latency as if they were the same hardware condition; use the `device`, version, commit, and contributor columns.
- A publishable stochastic-channel result should repeat multiple seeded trials. Use a separate run ID per trial or put the trial seed in the operating point.